# แยกเฟรมจากวิดีโอบน Google Colab

เชื่อม Google Drive แล้วแก้ `VIDEO_PATH` ให้ตรงกับวิดีโอที่ต้องการแยกเฟรม ก่อนรันเซลล์ถัดไป

เก็บทุก 150 เฟรม (ประมาณ 5 วินาทีเมื่อวิดีโอ 30 FPS) ภาพ JPG ใช้ขนาดเดิมของวิดีโอและเก็บใน `MyDrive/extracted_frames/<ชื่อวิดีโอ>`

สำหรับอีกมุมกล้อง ให้เปลี่ยน `VIDEO_PATH` แล้วรันเซลล์แยกเฟรมอีกครั้ง โฟลเดอร์ปลายทางจะเปลี่ยนตามชื่อวิดีโอ


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import cv2
from pathlib import Path

VIDEO_PATH = "/content/drive/MyDrive/Video_PRO/frontend.mp4"  # แก้พาธวิดีโอ
OUTPUT_ROOT = "/content/drive/MyDrive/extracted_frames"
FRAME_INTERVAL = 150  # เก็บภาพทุก 150 เฟรม

if FRAME_INTERVAL < 1:
    raise ValueError("FRAME_INTERVAL ต้องมากกว่า 0")

video_name = Path(VIDEO_PATH).stem
output_dir = Path(OUTPUT_ROOT) / video_name
output_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    cap.release()
    raise RuntimeError(f"เปิดวิดีโอไม่ได้ ตรวจพาธไฟล์: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_index = 0
saved_count = 0

try:
    while True:
        success, frame = cap.read()
        if not success:
            break

        if frame_index % FRAME_INTERVAL == 0:
            filename = output_dir / f"{video_name}_frame_{frame_index:06d}.jpg"
            if not cv2.imwrite(str(filename), frame, [cv2.IMWRITE_JPEG_QUALITY, 95]):
                raise IOError(f"บันทึกภาพไม่ได้: {filename}")
            saved_count += 1

        frame_index += 1
finally:
    cap.release()

print(f"บันทึกแล้ว: {saved_count} ภาพ")
if fps > 0:
    print(f"FPS: {fps:.2f} | เก็บภาพทุกประมาณ {FRAME_INTERVAL / fps:.2f} วินาที")
print("โฟลเดอร์ภาพ:", output_dir)
